# Pre-flight: does each parameter reach the model?

Run before any sweep. Perturbs one parameter at a time from a baseline and checks the ice
temperature actually changes. Four IDL runs.

A parameter can be declared in `settings.pro`, read from the calibration file, and still never
reach the physics. That failure is silent — the run exits 0 and writes plausible output. It has
happened five times in this project: unwired parameters, the spin-up running before the override,
reused `.dat` columns, `settings.pro` overriding the config, and the flow model never starting.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from icetemp.calibration.data import DataHandler
from icetemp.calibration.priors import Priors, PARAM_NAMES
from icetemp.calibration.runner import GloGEMRunner

BASELINE = (0.5, 1.0, 1.0)      # (f_ref, f_ins, f_adv) -- mid-range, physical defaults
PERTURB = {'refreeze_frac': 0.95, 'insul_scale': 1.8, 'advection_scale': 1.6}
TOL = 1e-4                       # degC; below IDL's 3-decimal print precision
LABELS = dict(zip(PARAM_NAMES, ('$f_{ref}$', '$f_{ins}$', '$f_{adv}$')))
print('PARAM_NAMES:', PARAM_NAMES, '\nbaseline:', BASELINE)

Test glacier: the Grenzgletscher band with the deepest borehole — thick ice, so all three parameters have room to act.

In [ ]:
dh = DataHandler(region='CentralEurope')
dh.load()
cands = [g for g in dh.calibration_glaciers if g.glacier_id and g.base_glacier_name == 'Grenzgletscher']
assert cands, 'no Grenzgletscher entity found'
glacier = max(cands, key=lambda g: g.depths.max())
print(f'{glacier.glacier_name}  id={glacier.glacier_id}  '
      f'elev={glacier.elevation:.0f} m  max observed depth={glacier.depths.max():.0f} m')

In [ ]:
# run_dir must be ABSOLUTE: run_design_point cds into GLOGEM_DIR before launching IDL, so a
# relative path would resolve somewhere else and silently fall through to the real config.pro.
runner = GloGEMRunner(
    run_dir=Path('../data/bayescal/_preflight/training_runs').resolve(),
    catchment='preflight_single',   # own catchment -- never touches the real ones
    thermal_spinup='y',             # match how calibration runs actually execute
)
runner.copy_calibration_data()
runner.write_catchment_file([glacier])
runner.write_glenglat_lookup([glacier])
print('config ->', runner.write_training_config())

## Run

`skip_if_done=False` forces a fresh run each time: GloGEM writes every run to the same filenames,
so a stale `.done` sentinel would make the parser read whichever run last touched disk.

In [ ]:
def run(tag, theta):
    ok = runner.run_design_point([glacier.glacier_id], theta, tag=tag, idl_bin='idl',
                                 timeout=1800, skip_if_done=False)
    assert ok, f'{tag} failed -- see {runner.run_dir / (tag + ".log")}'
    got = runner.parse_training_output([glacier]).get(glacier.glacier_name)
    assert got is not None, f'{tag} produced no profile for {glacier.glacier_name}'
    return got


depths, T_base = run('preflight_baseline', BASELINE)
profiles = {}
for k, name in enumerate(PARAM_NAMES):
    theta = list(BASELINE)
    theta[k] = PERTURB[name]
    profiles[name] = run(f'preflight_{name}', tuple(theta))[1]
    print(f'{name:16s} {BASELINE[k]} -> {PERTURB[name]}   done')

## Verdict

In [ ]:
depths = np.asarray(depths, dtype=float)
resolved = np.isfinite(T_base)
rows = []
for name in PARAM_NAMES:
    d = np.asarray(profiles[name], dtype=float) - np.asarray(T_base, dtype=float)
    m = resolved & np.isfinite(np.asarray(profiles[name], dtype=float))
    peak = float(np.nanmax(np.abs(d[m]))) if m.any() else np.nan
    z_at = float(depths[m][np.nanargmax(np.abs(d[m]))]) if m.any() else np.nan
    rows.append({'parameter': name, 'from': BASELINE[PARAM_NAMES.index(name)],
                 'to': PERTURB[name], 'max |dT| [degC]': peak,
                 'at depth [m]': z_at, 'reaches model': 'YES' if peak > TOL else 'NO'})

verdict = pd.DataFrame(rows)
display(verdict.round(4))
print(f'resolved depths: {resolved.sum()} of {len(depths)} '
      f'(grid extends to {depths.max():.0f} m; deeper levels are NaN = below the ice column)')

failed = verdict.loc[verdict['reaches model'] == 'NO', 'parameter'].tolist()
if failed:
    print(f'\nFAIL: {failed} did not change the profile. Check the read/apply path in '
          'read_firnicetemp_calibration.pro and apply_firnicetemp_calibration.pro, and that the '
          'override is applied BEFORE the spin-up runs (glogem.pro).')
else:
    print('\nPASS: all three parameters measurably change the ice temperature.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
axes[0].plot(np.asarray(T_base)[resolved], depths[resolved], 'k-', lw=2, label='baseline')
for name in PARAM_NAMES:
    T = np.asarray(profiles[name], dtype=float)
    axes[0].plot(T[resolved], depths[resolved], lw=1.4, label=LABELS[name])
    axes[1].plot((T - np.asarray(T_base, dtype=float))[resolved], depths[resolved],
                 lw=1.6, label=LABELS[name])
axes[1].axvline(0, color='k', lw=.8)
axes[0].set_xlabel('T [degC]')
axes[1].set_xlabel('change from baseline [degC]')
axes[0].set_ylabel('depth [m]')
for ax in axes:
    ax.invert_yaxis()
    ax.legend(fontsize=8)
axes[0].set_title('profiles', fontsize=10)
axes[1].set_title('effect of each parameter', fontsize=10)
fig.tight_layout()